# Solar filament segmentation: end-to-end pipeline

This notebook rebuilds the submitted predictions from the stored weights, without any training:

1. **Preprocessing**: limb fit (circle search on an edge map), limb-darkening-corrected contrast and
   distance-from-centre channels (`filseg.prep`, `filseg.solar`).
2. **Networks**: two U-Nets (EfficientNetV2-S encoder) trained on 5 time-grouped folds at 1024 and
   1536 px; each test image is predicted by all 10 fold models with 8-view dihedral test-time
   augmentation (`weights/<run>/fold{k}.pt`).
3. **Instances**: the two probability maps are averaged at full resolution (2048 px), thresholded
   with hysteresis, restricted to the solar disk, and fragments within 8 px are linked
   (`filseg.instances`).
4. **Keep rule**: a LightGBM model predicts each candidate's expected PQ contribution; candidates
   above the out-of-fold-tuned threshold are kept (`weights/keep_rule.txt`, `weights/postprocess.json`).
5. **Submission**: one compressed COCO RLE per filament -> `submission.csv`.

Inputs: the competition images (`MAGFiLO_1.0_Kaggle_2026/`; set `FILSEG_DATA_ROOT` if they are not in
`data/` or `/kaggle/input`). A GPU is recommended (about 45 min on one T4); the code also runs on CPU.

In [ ]:
import os, sys, time, json, platform
from importlib import metadata
from pathlib import Path

# repository root = the folder holding weights/postprocess.json (this notebook lives in notebooks/)
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "weights" / "postprocess.json").exists())
sys.path.insert(0, str(REPO / "src"))
os.environ["PYTHONPATH"] = str(REPO / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")  # for worker processes

import numpy as np
import pandas as pd
import torch
from filseg import data, pipeline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("python", platform.python_version(), "| torch", torch.__version__, "| device", device,
      torch.cuda.get_device_name(0) if device.type == "cuda" else "")


def version(pkg):
    try:
        return metadata.version(pkg)
    except metadata.PackageNotFoundError:
        return None


print({k: version(k) for k in ("numpy", "pandas", "scipy", "opencv-python", "opencv-python-headless", "pycocotools",
                               "lightgbm", "segmentation-models-pytorch", "timm", "torch") if version(k)})
print("data:", data.DATA_ROOT)

In [ ]:
post, keep_model = pipeline.load_post(REPO / "weights")
print(json.dumps({k: v for k, v in post.items() if k != "feature_columns"}, indent=1))
test_stems = sorted(p.stem for p in data.TEST_IMAGES.glob("*.jpeg"))
print("test images:", len(test_stems))

## Network predictions (10 fold models x 8 views)

In [ ]:
maps, disks = [], None
for run in post["runs"]:
    t0 = time.time()
    m, d = pipeline.probability_maps(test_stems, "test", run, REPO / "weights" / run["name"], device)
    maps.append(m)
    disks = d if disks is None else disks
    print(f"{run['name']}: maps {m.shape} in {(time.time() - t0) / 60:.1f} min")

## Instances, keep rule and submission

In [ ]:
t0 = time.time()
cons = pipeline.contrast_maps(test_stems, "test", disks)
sub = pipeline.make_submission(test_stems, maps, disks, cons, post, keep_model)
sub.to_csv("submission.csv", index=False)
n_img = sub.filament_id.str.rsplit("_", n=1).str[0].nunique()
print(f"{len(sub)} filaments on {n_img} of {len(test_stems)} test images in {(time.time() - t0) / 60:.1f} min -> submission.csv")

In [ ]:
# format checks: unique ids, ids map to test images, every mask decodes to 2048 x 2048 with positive area, no overlaps
from pycocotools import mask as mask_utils
chk = pd.read_csv("submission.csv")
assert list(chk.columns) == ["filament_id", "segmentation_rle"] and chk.filament_id.is_unique
assert set(chk.filament_id.str.rsplit("_", n=1).str[0]) <= set(test_stems)
for _stem, g in chk.groupby(chk.filament_id.str.rsplit("_", n=1).str[0]):
    m = mask_utils.decode([{"size": [data.H, data.W], "counts": c} for c in g.segmentation_rle])
    assert (m.reshape(-1, m.shape[-1]).sum(0) > 0).all() and m.sum(-1).max() <= 1
print("submission.csv passes format checks")

## Training (not run here)

The weights were produced on Kaggle (2 x T4) with `filseg.train.run`, e.g.

```python
from filseg.train import Config, run
run(Config(size=1024, epochs=40, seed=0))                 # weights/unet_effv2s_1024
run(Config(size=1536, epochs=60, seed=1, eval_every=10))  # weights/unet_effv2s_1536
```

and the keep rule with the post-processing notebook (`dev/build_post_notebook.py`); see the README.